<a href="https://colab.research.google.com/github/NiharMarar/Portfolio2/blob/main/Animal_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from sklearn.utils import class_weight

In [12]:
def preprocess(image, label):
  return tf.image.resize(image, (64, 64)) / 255.0, label

train_data, test_data = tfds.load("cats_vs_dogs", split=["train[:80%]", "train[80%:]"], as_supervised=True)
train_data = train_data.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)
test_data = test_data.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(32, (3, 3), activation = "relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(2, activation = "softmax")
])

model.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"])

In [14]:
labels = np.concatenate([y for x, y in train_data], axis=0)
class_weight_dict = dict(enumerate(class_weight.compute_class_weight('balanced', classes=np.unique(labels), y=labels)))

In [ ]:
model.fit(train_data, validation_data=test_data, epochs=2, class_weight=class_weight_dict)

Epoch 1/2
582/582 ━━━━━━━━━━━━━━━━━━━━ 84s 140ms/step - accuracy: 0.5519 - loss: 0.6829 - val_accuracy: 0.6150 - val_loss: 0.6511
Epoch 2/2


In [ ]:
from tensorflow.keras.preprocessing import image

image_path = ""
img = image.load_img(image_path, target_size = (64, 64))
img_array = np.expand_dims(image.img_to_array(img), axis=0) / 255.0

prediction = model.predict(img_array)
label = np.argmax(prediction)

if label == 0:
    print("It's a Dog!")
else:
    print("It's a Cat!")
